# Notebook 2 — Batch ingestion & Parquet

Reads raw CSV from HDFS, cleans types, expands Last.fm to ~5M timestamped events, writes partitioned Parquet.


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))
import hdfs_paths as hp

from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, year, to_date, regexp_replace, coalesce
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType, DateType,
)

spark = (
    SparkSession.builder.appName("MusicTrend_02_Ingestion")
    .config("spark.sql.shuffle.partitions", hp.SHUFFLE_PARTITIONS)
    .getOrCreate()
)


## Synthetic expansion methodology

Each Last.fm row `(userID, artistID, weight)` is interpreted as a **play count**. We synthesize that many events (with Gaussian noise on the count, σ = 0.1×weight) and assign timestamps uniformly at random in a **730-day** window starting **2019-01-01**. This preserves aggregate listening intensity per user–artist pair while yielding event-level data suitable for windowed/streaming workloads. Distributions of per-artist totals remain aligned with the source weights up to sampling noise.


In [ ]:
# Read Last.fm with explicit schema
raw_lastfm = (
    spark.read.schema(
        StructType(
            [
                StructField("userID", StringType(), True),
                StructField("artistID", StringType(), True),
                StructField("weight", StringType(), True),
            ]
        )
    )
    .option("sep", "\t")
    .option("header", False)
    .csv(hp.RAW_LASTFM)
)


def expand_partition(rows):
    import numpy as np
    from datetime import datetime, timedelta

    BASE = datetime(2019, 1, 1)
    WINDOW = 730 * 86400
    rng = np.random.default_rng()
    for row in rows:
        w = float(row.weight) if row.weight is not None else 1.0
        weight = max(1, int(w))
        n = max(1, int(weight + rng.normal(0, weight * 0.1)))
        offsets = rng.integers(0, WINDOW, size=n, endpoint=False)
        for o in offsets:
            ts = BASE + timedelta(seconds=int(o))
            yield Row(
                user_id=str(row.userID),
                artist_id=str(row.artistID),
                event_timestamp=ts.strftime("%Y-%m-%d %H:%M:%S"),
            )


expanded_rdd = raw_lastfm.rdd.mapPartitions(expand_partition)
events_df = spark.createDataFrame(expanded_rdd).withColumn(
    "event_timestamp", col("event_timestamp").cast(TimestampType())
).withColumn("year", year("event_timestamp"))

ec = events_df.count()
print(f"Expanded event count: {ec}")
events_df.write.mode("overwrite").partitionBy("year").parquet(hp.PROCESSED_EVENTS)


In [ ]:
# Spotify charts — clean & partition by region
spotify_raw = spark.read.schema(
    StructType(
        [
            StructField("title", StringType(), True),
            StructField("rank", IntegerType(), True),
            StructField("date", StringType(), True),
            StructField("artist", StringType(), True),
            StructField("url", StringType(), True),
            StructField("region", StringType(), True),
            StructField("chart", StringType(), True),
            StructField("trend", StringType(), True),
            StructField("streams", LongType(), True),
        ]
    )
).option("header", True).csv(hp.RAW_SPOTIFY)

spotify_df = (
    spotify_raw.withColumn(
        "chart_date",
        coalesce(
            to_date(col("date"), "yyyy-MM-dd"),
            to_date(col("date"), "M/d/yyyy"),
            to_date(col("date")),
        ),
    )
    .withColumn("streams", col("streams").cast(LongType()))
    .select("title", "rank", "chart_date", "artist", "region", "streams")
    .filter(col("chart_date").isNotNull())
)
spotify_df.write.mode("overwrite").partitionBy("region").parquet(hp.PROCESSED_SPOTIFY)
print("Spotify rows:", spotify_df.count())


In [ ]:
# Billboard
bb_raw = spark.read.schema(
    StructType(
        [
            StructField("WeekID", StringType(), True),
            StructField("Song", StringType(), True),
            StructField("Performer", StringType(), True),
            StructField("SongID", StringType(), True),
            StructField("Instance", StringType(), True),
            StructField("Previous Week Position", StringType(), True),
            StructField("Peak Position", StringType(), True),
            StructField("Weeks on Chart", StringType(), True),
        ]
    )
).option("header", True).csv(hp.RAW_BILLBOARD)

billboard_df = (
    bb_raw.select(
        coalesce(
            to_date(col("WeekID"), "M/d/yyyy"),
            to_date(col("WeekID"), "yyyy-MM-dd"),
            to_date(col("WeekID")),
        ).alias("week_date"),
        col("Song").alias("song"),
        col("Performer").alias("performer"),
        col("SongID").alias("song_id"),
        regexp_replace(col("Peak Position"), "[^0-9]", "").cast(IntegerType()).alias("peak_position"),
        regexp_replace(col("Weeks on Chart"), "[^0-9]", "").cast(IntegerType()).alias("weeks_on_chart"),
    )
    .filter(col("week_date").isNotNull())
)
billboard_df.write.mode("overwrite").parquet(hp.PROCESSED_BILLBOARD)
print("Billboard rows:", billboard_df.count())


In [ ]:
# MSD audio features (subset schema)
msd_raw = spark.read.schema(
    StructType(
        [
            StructField("artist_name", StringType(), True),
            StructField("song_title", StringType(), True),
            StructField("tempo", DoubleType(), True),
            StructField("energy", DoubleType(), True),
            StructField("loudness", DoubleType(), True),
            StructField("danceability", DoubleType(), True),
            StructField("key", StringType(), True),
            StructField("mode", StringType(), True),
        ]
    )
).option("header", True).csv(hp.RAW_MSD)

audio_df = msd_raw.select(
    col("artist_name"),
    col("tempo").cast(DoubleType()),
    col("energy").cast(DoubleType()),
    col("loudness").cast(DoubleType()),
    col("danceability").cast(DoubleType()),
)
audio_df.write.mode("overwrite").parquet(hp.PROCESSED_AUDIO)
print("MSD audio rows:", audio_df.count())


In [ ]:
# Verification reads
for label, path in [
    ("events", hp.PROCESSED_EVENTS),
    ("spotify", hp.PROCESSED_SPOTIFY),
    ("billboard", hp.PROCESSED_BILLBOARD),
    ("audio", hp.PROCESSED_AUDIO),
]:
    df = spark.read.parquet(path)
    print(label, "partitions:", df.rdd.getNumPartitions(), "rows:", df.count())


## Outputs confirmed

- Expanded events written under `processed/events/` partitioned by `year`.
- Spotify, Billboard, MSD audio written as Parquet per spec.
- Row totals printed after write.
